In [1]:
import glob
import pandas as pd

def load_wisdm_from_files(path):

    files = glob.glob(path + "/*")
    files = [f for f in files if "data_" in f]

    print("Total files:", len(files))

    all_data = []

    for file in files:
        try:
            # ✅ correct separator = comma
            df = pd.read_csv(file, sep=",", header=None)

            # assign columns
            df.columns = ["user", "activity", "timestamp", "x", "y", "z"]

            # remove semicolon from z
            df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)

            # convert to numeric
            df["x"] = pd.to_numeric(df["x"], errors="coerce")
            df["y"] = pd.to_numeric(df["y"], errors="coerce")
            df["z"] = pd.to_numeric(df["z"], errors="coerce")

            df.dropna(inplace=True)

            all_data.append(df)

        except Exception as e:
            print(f"Skipping {file}: {e}")

    data = pd.concat(all_data, ignore_index=True)
    return data


DATA_PATH = "wisdm-dataset/wisdm-dataset/raw/phone/accel"
data = load_wisdm_from_files(DATA_PATH)

print(data.shape)
print(data.head())

Total files: 51
(4804403, 6)
   user activity        timestamp         x          y         z
0  1600        A  252207666810782 -0.364761   8.793503  1.055084
1  1600        A  252207717164786 -0.879730   9.768784  1.016998
2  1600        A  252207767518790  2.001495  11.109070  2.619156
3  1600        A  252207817872794  0.450623  12.651642  0.184555
4  1600        A  252207868226798 -2.164352  13.928436 -4.422485


In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data["activity"] = le.fit_transform(data["activity"])

In [3]:
X_raw = data[["x", "y", "z"]].values
y_raw = data["activity"].values

In [4]:
import numpy as np
def create_windows(X, y, window_size=128, step=100):
    Xs, ys = [], []

    for i in range(0, len(X) - window_size, step):
        Xs.append(X[i:i+window_size])
        ys.append(y[i + window_size // 2])

    return np.array(Xs), np.array(ys)

X, y = create_windows(X_raw, y_raw)

print(X.shape)  # expected (~24000+, 200, 3)

(48043, 128, 3)


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X.reshape(-1, X.shape[2])).reshape(X.shape)

In [6]:
def downsample_naive(X, factor=2):
    return X[:, ::factor, :]

def downsample_mean(X, factor=2):
    N, T, F = X.shape
    T_new = T // factor
    
    X = X[:, :T_new*factor, :]
    X = X.reshape(N, T_new, factor, F)
    
    return X.mean(axis=2)

In [7]:
from scipy.signal import butter, filtfilt, savgol_filter

def butter_filter(X, cutoff=5, fs=100, order=3):
    b, a = butter(order, cutoff / (0.5 * fs), btype='low')
    X_filtered = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[2]):
            X_filtered[i, :, j] = filtfilt(b, a, X[i, :, j])
    
    return X_filtered


def savgol_filtering(X, window=7, poly=2):
    X_filtered = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[2]):
            X_filtered[i, :, j] = savgol_filter(X[i, :, j], window, poly)
    
    return X_filtered

In [8]:
from sklearn.preprocessing import StandardScaler

def normalize_data(X):
    N, T, F = X.shape
    X_flat = X.reshape(-1, F)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_flat)

    return X_scaled.reshape(N, T, F)

In [9]:
X = normalize_data(X)

In [10]:
def extract_features(X):
    mean = np.mean(X, axis=1)
    std = np.std(X, axis=1)
    max_val = np.max(X, axis=1)
    min_val = np.min(X, axis=1)

    return np.concatenate([mean, std, max_val, min_val], axis=1)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, MaxPooling1D

def build_tiny_cnn(input_shape, num_classes):

    model = Sequential([
        Conv1D(16, 3, activation='relu', input_shape=input_shape),
        MaxPooling1D(2),

        Conv1D(32, 3, activation='relu'),
        MaxPooling1D(2),

        Conv1D(32, 3, activation='relu'),

        Flatten(),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def get_models():
    return {
        "LogisticRegression": LogisticRegression(max_iter=200),
        "DecisionTree": DecisionTreeClassifier(max_depth=15, min_samples_split=10),
        "TinyCNN": "cnn"
    }

In [13]:
import time
from sklearn.metrics import accuracy_score
import numpy as np

def train_and_evaluate(X_train, y_train, X_test, y_test, model):

    input_shape = X_train.shape

    # 🔷 CNN case
    if model == "cnn":

        num_classes = len(np.unique(y_train))

        cnn = build_tiny_cnn(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            num_classes=num_classes
        )

        # 🔹 Training
        start_train = time.time()
        cnn.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
        train_time = time.time() - start_train

        # 🔹 Inference
        start_inf = time.time()
        y_pred = np.argmax(cnn.predict(X_test, verbose=0), axis=1)
        inference_time = time.time() - start_inf

    else:
        # 🔷 sklearn models (LR + DT)
        from sklearn.base import clone
        model_copy = clone(model)

        # X_train_flat = X_train.reshape(X_train.shape[0], -1)
        # X_test_flat = X_test.reshape(X_test.shape[0], -1)
        X_train_flat = extract_features(X_train)
        X_test_flat = extract_features(X_test)

        start_train = time.time()
        model_copy.fit(X_train_flat, y_train)
        train_time = time.time() - start_train

        start_inf = time.time()
        y_pred = model_copy.predict(X_test_flat)
        inference_time = time.time() - start_inf

    # 🔹 Accuracy
    acc = accuracy_score(y_test, y_pred)

    # 🔹 Per sample inference
    inf_per_sample = inference_time / len(X_test)

    return {
        "accuracy": acc,
        "train_time": train_time,
        "inference_time": inference_time,
        "inference_time/sample": inf_per_sample,
        "input_shape": input_shape
    }

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
def run_all_experiments(X, y):

    results = []
    models = get_models()

    from sklearn.model_selection import train_test_split
    from sklearn.base import clone

    # Precompute
    X_naive = downsample_naive(X)
    X_mean = downsample_mean(X)
    X_butter = butter_filter(X)
    X_savgol = savgol_filtering(X)

    pipelines = [
        ("Raw", X, "No Processing"),

        ("Naive Downsample", X_naive, "Downsampling Only"),
        ("Mean Downsample", X_mean, "Downsampling Only"),

        ("Butterworth", X_butter, "Filtering Only"),
        ("Savitzky-Golay", X_savgol, "Filtering Only"),

        ("Naive + Butterworth", butter_filter(X_naive), "Downsampling → Filtering"),
        ("Mean + Butterworth", butter_filter(X_mean), "Downsampling → Filtering"),
        ("Naive + Savgol", savgol_filtering(X_naive), "Downsampling → Filtering"),
        ("Mean + Savgol", savgol_filtering(X_mean), "Downsampling → Filtering"),

        ("Butterworth → Naive", downsample_naive(X_butter), "Filtering → Downsampling"),
        ("Butterworth → Mean", downsample_mean(X_butter), "Filtering → Downsampling"),
        ("Savgol → Naive", downsample_naive(X_savgol), "Filtering → Downsampling"),
        ("Savgol → Mean", downsample_mean(X_savgol), "Filtering → Downsampling"),
    ]

    for model_name, model in models.items():

        print(f"\nRunning for model: {model_name}")

        for name, X_proc, pipeline_type in pipelines:

            X_train, X_test, y_train, y_test = train_test_split(
                X_proc, y, test_size=0.2, random_state=42
            )

            if model == "cnn":
                metrics = train_and_evaluate(X_train, y_train, X_test, y_test, model)
            else:
                from sklearn.base import clone
                model_copy = clone(model)
                metrics = train_and_evaluate(X_train, y_train, X_test, y_test, model_copy)

            metrics["name"] = name
            metrics["pipeline_type"] = pipeline_type
            metrics["model"] = model_name

            results.append(metrics)

    return pd.DataFrame(results)

In [16]:
df_results = run_all_experiments(X, y)


Running for model: LogisticRegression


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/m


Running for model: DecisionTree

Running for model: TinyCNN


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(sha

In [17]:
def compute_score(row):
    return (
        row["accuracy"]
        - 0.1 * row["inference_time"]
        - 0.000001 * row["train_time"]
    )

In [18]:
df_results["score"] = df_results.apply(compute_score, axis=1)

In [19]:
def format_results(df):
    df = df.copy()

    # reorder columns EXACTLY like your desired format
    df = df[
        [
            "model",
            "name",
            "pipeline_type",
            "accuracy",
            "train_time",
            "inference_time",
            "inference_time/sample",
            "input_shape",
            "score",
        ]
    ]

    # round values for neat printing
    df["accuracy"] = df["accuracy"].round(6)
    df["train_time"] = df["train_time"].round(6)
    df["inference_time"] = df["inference_time"].round(6)
    df["inference_time/sample"] = df["inference_time/sample"].apply(lambda x: f"{x:.6e}")
    df["score"] = df["score"].round(6)

    return df

In [20]:
models = df_results["model"].unique()

for m in models:
    print("\n" + "=" * 60)
    print(f"🔹 MODEL: {m}")
    print("=" * 60)

    df_model = df_results[df_results["model"] == m]

    # ✅ sort by score (descending)
    df_model = df_model.sort_values(by="score", ascending=False)

    # ✅ format nicely
    df_model = format_results(df_model)

    # ✅ print without index
    print(df_model.to_string(index=False))


🔹 MODEL: LogisticRegression
             model                name            pipeline_type  accuracy  train_time  inference_time inference_time/sample     input_shape    score
LogisticRegression      Savgol → Naive Filtering → Downsampling  0.257051    3.185060        0.000000          0.000000e+00  (38434, 64, 3) 0.257047
LogisticRegression     Mean Downsample        Downsampling Only  0.256530    3.025323        0.001525          1.587224e-07  (38434, 64, 3) 0.256375
LogisticRegression      Savitzky-Golay           Filtering Only  0.256530    3.062767        0.005394          5.613469e-07 (38434, 128, 3) 0.255988
LogisticRegression       Savgol → Mean Filtering → Downsampling  0.255906    3.046120        0.001999          2.080239e-07  (38434, 64, 3) 0.255703
LogisticRegression    Naive Downsample        Downsampling Only  0.251327    2.957286        0.000000          0.000000e+00  (38434, 64, 3) 0.251324
LogisticRegression                 Raw            No Processing  0.249766    